## Stream Customers Data From Cloud Files to Delta Lake using Auto Loader
1. Read files from cloud storage using Auto Loader
1. Transform the dataframe to add the following columns
    -   file path: Cloud file path
    -   ingestion date: Current Timestamp
1. Write the transformed data stream to Delta Lake Table

Autoloader in Databricks is a feature used to automatically ingest new files from cloud storage into a table or DataFrame in a reliable and scalable way. It is mainly used for streaming data ingestion in data engineering pipelines.
Autoloader automatically detects new files arriving in cloud storage (like S3, ADLS, GCS) and loads them into Delta tables or Spark DataFrames.

### 1. Read files using Auto Loader

In [0]:
customers_df =(spark.readStream
               .format("cloudFiles")
               .option("cloudFiles.format","json")
               .option("cloudfiles.schemaLocation","/Volumes/ginzobox/landing/operational_data/customer_autoloader/_schema")
               .option("cloudFiles.inferColumnTypes",'true')
               .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP")
               .load("/Volumes/ginzobox/landing/operational_data/customer_autoloader/"))

### 2. Transform the dataframe to add the following columns
- file path: Cloud file path
- ingestion date: Current Timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, col

customers_transformed_df = (
                                customers_df.withColumn("file_path", col("_metadata.file_path"))
                                            .withColumn("ingestion_date", current_timestamp())
)

### 3. Write the transformed data stream to Delta Table 

In [0]:
streaming_query = (
                    customers_transformed_df.writeStream
                        .format("delta")
                        .option("checkpointLocation", "/Volumes/ginzobox/landing/operational_data/customers_autoloader/_checkpoint_stream")
                        .toTable("ginzobox.bronze.customers_autoloader")
)

In [0]:
streaming_query.stop()

In [0]:
%sql
SELECT * FROM ginzobox.bronze.customers_autoloader;